# M2_dr vs M2_ft — SNR + per-band comparison

Compares the **band/rate-invariant** fine-tune (`M2_dr`, windowed) against the original single-rate
`M2_ft` (no-window). Each model is run with **its own** preprocessing (FFT window + db_vmin/vmax) and
its **val-tuned decision threshold** — that's essential, M2_dr's operating point is ~0.95, M2_ft ~0.85.

Sections: (1) threshold sweep, (2) detection vs SNR/attenuation, (3) rate-invariance, (4) per-band
detection on live captures, (5) the sweep's measured PSD per band.


In [ ]:
import os, sys, json, glob, numpy as np, torch, matplotlib.pyplot as plt
from pathlib import Path
%matplotlib inline
REPO = Path('/home/sat3737/holohub-dev/dino_fine_tuning')   # <-- edit if elsewhere
sys.path.insert(0, str(REPO/'src'))
d = os.environ.get('DINOV3_REPO')   # DINOv3 backbone repo must be importable
if d and d not in sys.path: sys.path.insert(0, d)
import rfdata as rf, rate_augment as ra
from model import DinoSegmenter

# --- model registry: each with its OWN window / vmin-vmax / tuned threshold ---
MODELS = {
  'M2_dr': dict(ckpt=REPO/'checkpoints/M2_dr/best.pt', window='hann',
                meta=REPO/'data/dataset_dr/dataset_meta.json', thr=0.95, color='C0'),
  'M2_ft': dict(ckpt=Path('/home/bqn82/Holohub-Signal-Detection/dino_fine_tuning/checkpoints/M2_ft/best.pt'),
                window='none',
                meta=Path('/home/bqn82/Holohub-Signal-Detection/dino_fine_tuning/data/dataset/dataset_meta.json'),
                thr=0.85, color='C1'),
}
NFFT, TILE = 1024, 256
def load(cfg):
    st=torch.load(cfg['ckpt'], map_location='cuda'); c=st['cfg']
    m=DinoSegmenter(c['weights_path'], feat_layers=tuple(c['feat_layers']), mode=st['mode'],
                    unfreeze_last_n=c['unfreeze_last_n']).cuda(); m.load_state_dict(st['model']); m.eval()
    mt=json.load(open(cfg['meta'])); cfg['vmin'], cfg['vmax'] = mt['db_vmin'], mt['db_vmax']; cfg['model']=m
    return cfg
for k in list(MODELS):
    try: MODELS[k]=load(MODELS[k]); print(f"loaded {k}: window={MODELS[k]['window']} vmin/vmax={MODELS[k]['vmin']:.1f}/{MODELS[k]['vmax']:.1f} thr={MODELS[k]['thr']}")
    except Exception as e: print(f'SKIP {k}: {e}'); MODELS.pop(k)

@torch.no_grad()
def predict(cfg, db):
    x=np.clip((db-cfg['vmin'])/max(cfg['vmax']-cfg['vmin'],1e-6),0,1).astype(np.float32)
    t=torch.from_numpy(x)[None,None].cuda() if x.ndim==2 else torch.from_numpy(x)[:,None].cuda()
    with torch.autocast('cuda',dtype=torch.bfloat16): pr=torch.sigmoid(cfg['model'](t).float())
    return pr.squeeze(1).cpu().numpy()
def dbframe(iq, window): return rf.frames_to_db(torch.from_numpy(iq[None]).cuda(),NFFT,TILE,window=window)[0].cpu().numpy()
def iou_pr(pred, gt):
    p=pred.astype(bool); g=gt.astype(bool); tp=int((p&g).sum()); fp=int((p&~g).sum()); fn=int((~p&g).sum())
    return tp,fp,fn


## 1. Threshold sweep (M2_dr on DR val) — confirm the operating point


In [ ]:
from dataset import RFSegDataset; from torch.utils.data import DataLoader, Subset
ds=RFSegDataset(str(REPO/'data/dataset_dr'),'val',augment=False)
idx=np.random.default_rng(0).choice(len(ds),2000,replace=False).tolist()
ths=np.round(np.arange(0.5,0.99,0.05),2)
agg={t:[0,0,0] for t in ths}
with torch.no_grad():
    for b in DataLoader(Subset(ds,idx),batch_size=16,num_workers=4):
        img=b['image'].cuda(); m=b['mask'].cuda().bool()
        with torch.autocast('cuda',dtype=torch.bfloat16): prob=torch.sigmoid(MODELS['M2_dr']['model'](img).float())
        for t in ths:
            p=prob>=t; a=agg[t]; a[0]+=int((p&m).sum()); a[1]+=int((p&~m).sum()); a[2]+=int((~p&m).sum())
iou=[agg[t][0]/(sum(agg[t])+1e-9) for t in ths]; prec=[agg[t][0]/(agg[t][0]+agg[t][1]+1e-9) for t in ths]
best=ths[int(np.argmax(iou))]; print('best-IoU threshold =', best)
fig,ax=plt.subplots(figsize=(7,4)); ax.plot(ths,iou,'o-',label='IoU'); ax.plot(ths,prec,'s-',label='precision')
ax.axvline(best,color='k',ls=':',label=f'best {best}'); ax.set_xlabel('threshold'); ax.legend(); ax.grid(alpha=.3)
ax.set_title('M2_dr val IoU/precision vs decision threshold'); plt.show()


## 2. Detection vs SNR (attenuation sweep)
Both models on the labeled captures at native 245.76 MS/s, each at its own window+threshold. Higher
attenuation = lower SNR. (These captures are the training source, so this is a *model-vs-model*
operating comparison, not held-out generalization — the DR val/test splits cover that.)


In [ ]:
caps = sorted(glob.glob('/home/bqn82/captures/attenuation_dB_*.sigmf-meta'))
caps = [c for c in caps if '.test.' not in c]
import re
def atten(p): m=re.search(r'attenuation_dB_(\d+)',p); return int(m.group(1)) if m else None
N_FR=12
res={k:{} for k in MODELS}
for cp in caps:
    a=atten(cp);
    if a is None: continue
    cap=rf.load_capture(Path(cp)); mm=cap.memmap(); fsamp=NFFT*TILE
    nf=cap.n_samples//fsamp
    if nf<2: continue
    starts=np.linspace(0,(nf-1)*fsamp,min(N_FR,nf)).astype(np.int64)
    per={k:[0,0,0] for k in MODELS}
    for s in starts:
        iq=np.array(mm[s:s+fsamp],np.complex64); gt,_=rf.build_frame_mask(cap,int(s),NFFT,TILE)
        for k,cfg in MODELS.items():
            db=dbframe(iq,cfg['window']); pr=predict(cfg,db)[0]>=cfg['thr']
            tp,fp,fn=iou_pr(pr,gt); per[k][0]+=tp; per[k][1]+=fp; per[k][2]+=fn
    for k in MODELS:
        tp,fp,fn=per[k]; res[k][a]=dict(iou=tp/(tp+fp+fn+1e-9),prec=tp/(tp+fp+1e-9),rec=tp/(tp+fn+1e-9))
fig,ax=plt.subplots(1,2,figsize=(14,4.5))
for k,cfg in MODELS.items():
    xs=sorted(res[k]); ax[0].plot(xs,[res[k][a]['iou'] for a in xs],'o-',color=cfg['color'],label=k)
    ax[1].plot(xs,[res[k][a]['prec'] for a in xs],'o-',color=cfg['color'],label=k)
ax[0].set_title('pixel IoU vs attenuation'); ax[1].set_title('precision vs attenuation')
for a in ax: a.set_xlabel('attenuation dB (higher = lower SNR)'); a.grid(alpha=.3); a.legend()
plt.show()


## 3. Rate-invariance (M2_dr) — the retrain's whole point
Emulate one labeled capture at several sample rates and run M2_dr; IoU should stay ~flat across rate.


In [ ]:
cap=rf.load_capture(Path('/home/bqn82/captures/attenuation_dB_20.sigmf-meta')); mm=cap.memmap(); src=cap.sample_rate
rates=[20.48e6,61.44e6,122.88e6,245.76e6]; cfg=MODELS['M2_dr']; out=[]
for R in rates:
    D=max(1,round(src/R)); need=NFFT*TILE*D; nf=cap.n_samples//need
    starts=np.linspace(0,(nf-1)*need,min(12,nf)).astype(np.int64); tp=fp=fn=0
    for s in starts:
        chunk=np.array(mm[s:s+need],np.complex64)
        db,gt,_=ra.emulate_frame(chunk,int(s),cap.annotations,src,R,0.0,NFFT,TILE,window='hann',device='cuda')
        pr=predict(cfg,db)[0]>=cfg['thr']; t,f,n=iou_pr(pr,gt); tp+=t; fp+=f; fn+=n
    out.append((R/1e6, tp/(tp+fp+fn+1e-9))); print(f'{R/1e6:7.2f} MS/s: IoU={out[-1][1]:.3f}')
plt.figure(figsize=(6,4)); plt.plot([r for r,_ in out],[i for _,i in out],'o-'); plt.ylim(0,1)
plt.xlabel('sample rate MS/s'); plt.ylabel('IoU'); plt.title('M2_dr IoU vs rate (attenuation_dB_20)'); plt.grid(alpha=.3); plt.show()


## 4. Per-band live detection
Run both models on a spectrogram from a live capture at each band; visualize input + each model's mask.
Add your band captures to `BAND_CAPS`. The 2400 ISM example uses a labeled capture; for **1150 quiet**
and **500 HDTV** you need short live captures at those centers (the sweep bursts are only 4 rows — too
short). Capture ~1 s of IQ at a band and drop the `.sigmf-meta` path in below.


In [ ]:
BAND_CAPS = {
  '2400 ISM (labeled)': '/home/bqn82/captures/attenuation_dB_20.sigmf-meta',
  # '1150 quiet': '/path/to/quiet_1150.sigmf-meta',
  # '500 HDTV':  '/path/to/hdtv_500.sigmf-meta',
}
for name,cp in BAND_CAPS.items():
    if not Path(cp).exists(): print(f'-- {name}: capture not found, skipping --'); continue
    cap=rf.load_capture(Path(cp)); mm=cap.memmap(); fsamp=NFFT*TILE
    s=int((cap.n_samples//fsamp//2))*fsamp; iq=np.array(mm[s:s+fsamp],np.complex64)
    gt,_=rf.build_frame_mask(cap,s,NFFT,TILE)
    cols=1+len(MODELS); fig,ax=plt.subplots(1,cols,figsize=(5*cols,4)); ax=np.atleast_1d(ax)
    base=MODELS[list(MODELS)[0]]; db0=dbframe(iq, base['window'])
    ax[0].imshow(np.clip((db0-base['vmin'])/max(base['vmax']-base['vmin'],1e-6),0,1),aspect='auto',cmap='viridis',vmin=0,vmax=1)
    if gt.sum()>0: ax[0].imshow(np.ma.masked_where(gt<0.5,gt),aspect='auto',cmap='autumn',alpha=0.3)
    ax[0].set_title(f'{name}\ninput + GT(red)')
    for j,(k,cfg) in enumerate(MODELS.items()):
        db=dbframe(iq,cfg['window']); pr=predict(cfg,db)[0]
        ax[j+1].imshow(pr,aspect='auto',cmap='inferno',vmin=0,vmax=1)
        ax[j+1].set_title(f'{k} prob (>{cfg["thr"]}: {100*(pr>=cfg["thr"]).mean():.1f}%)')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()


## 5. Sweep live PSD per band
The measured spectrum (antenna, ch0) at swept centers — shows which bands are occupied vs quiet, from
the real captured `sweep_psd.npz` (this is what the sweep is good for; it's too short to feed the model).


In [ ]:
SWEEP='/home/bqn82/captures/dino_finetune_x411_sweep/low/sweep_psd.npz'
if Path(SWEEP).exists():
    z=np.load(SWEEP); c=z['center_hz']; r=z['rate_hz']; g=z['gain_db']; psd=z['psd_db']
    rate=245.76e6; gsel=np.median(np.unique(g))
    targets=[c[np.argmin(np.abs(c-t))] for t in (1150e6,2400e6,500e6,3500e6)]
    fig,ax=plt.subplots(figsize=(10,4))
    for tc in sorted(set(targets)):
        sel=(np.abs(c-tc)<1)&(np.abs(r-rate)<1e6)&(np.abs(g-gsel)<3)
        if sel.any():
            fax=np.linspace(-rate/2,rate/2,psd.shape[1])/1e6
            ax.plot(fax, psd[sel].mean(0), lw=1, label=f'{tc/1e6:.0f} MHz')
    ax.set_xlabel('baseband freq (MHz)'); ax.set_ylabel('power dB'); ax.grid(alpha=.3); ax.legend()
    ax.set_title(f'sweep antenna PSD @ {rate/1e6:.0f} MS/s (gain~{gsel:.0f} dB) per band'); plt.show()
else: print('sweep_psd.npz not found at', SWEEP)
